# EAT Deep Segmentation - Colab notebook
Deep Learning-Based Automated Segmentation and Quantification of Epicardial Adipose Tissue from Cardiac CT Images

**Before running:** Runtime -> Change runtime type -> **GPU**. Then run the cells from top to bottom. All outputs stay visible in this notebook, so you can show it in the review.

## 1. Upload and extract the project

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose EAT_Deep_Segmentation.zip
!unzip -q -o EAT_Deep_Segmentation.zip
%cd EAT_Deep_Segmentation
!pip -q install opencv-python-headless pandas scikit-learn

## 2. Upload the dataset zip and convert it to PNG slices
Dataset: Siriapisith et al. (2021), figshare DOI 10.6084/m9.figshare.17039813 (NRRD volumes + EAT ground truth).

In [ ]:
uploaded = files.upload()          # choose the dataset zip (figshare EAT dataset, NRRD format)
zip_name = list(uploaded.keys())[0]
!mkdir -p /content/eat_data
!unzip -q -o "{zip_name}" -d /content/eat_data
!pip -q install pynrrd
!python src/prepare_nrrd_dataset.py --src /content/eat_data --inspect
# after checking the output above, create the training slices:
!python src/prepare_nrrd_dataset.py --src /content/eat_data --filter train_NC --patients 12 --slices 8 --clear

## 3. Train the U-Net

In [ ]:
!python src/train.py

## 4. Training curves

In [ ]:
from IPython.display import Image, display
for f in ['loss_curve','dice_curve','iou_curve']:
    display(Image(f'outputs/graphs/{f}.png'))

## 5. Evaluate on the test images

In [ ]:
!python src/evaluate.py
import pandas as pd
pd.read_csv('outputs/results.csv').tail(10)

## 6. Prediction images (original | ground truth | predicted | overlay)

In [ ]:
import glob
for f in sorted(glob.glob('outputs/predictions/*_result.png'))[:5]:
    display(Image(f))

## 7. Demo: one image in -> EAT mask and area out

In [ ]:
import glob, os
sample = sorted(glob.glob('data/images/*.png'))[0]
!python src/predict.py --image {sample} --no-show
display(Image('outputs/predictions/' + os.path.splitext(os.path.basename(sample))[0] + '_demo.png'))

## 8. Save everything (model + outputs) to Google Drive / download

In [ ]:
!zip -rq EAT_results.zip models outputs data/pairs.csv
files.download('EAT_results.zip')